# OMOP CDM Metadata and Infrastructure Tables

Creates supporting OMOP tables for metadata, locations, and observation periods.

## Tables

| Table | Purpose |
|-------|--------|
| cdm_source | CDM source metadata |
| location | Geographic locations |
| observation_period | Patient observation periods |
| death | Patient death records |

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## CDM_SOURCE Table

Contains metadata about the CDM instance.

In [ ]:
CREATE TABLE IF NOT EXISTS cdm_source (
  cdm_source_name STRING NOT NULL COMMENT 'Name of the CDM source'
  ,cdm_source_abbreviation STRING NOT NULL COMMENT 'Abbreviation'
  ,cdm_holder STRING COMMENT 'Organization holding the CDM'
  ,source_description STRING COMMENT 'Description of source data'
  ,source_documentation_reference STRING COMMENT 'Link to documentation'
  ,cdm_etl_reference STRING COMMENT 'Link to ETL documentation'
  ,source_release_date DATE COMMENT 'Source data release date'
  ,cdm_release_date DATE COMMENT 'CDM release date'
  ,cdm_version STRING COMMENT 'CDM version (e.g., v5.4)'
  ,cdm_version_concept_id BIGINT COMMENT 'CDM version concept'
  ,vocabulary_version STRING COMMENT 'Vocabulary version'
)
USING DELTA
COMMENT 'OMOP CDM Source metadata table'
TBLPROPERTIES ('quality' = 'gold');

In [ ]:
-- Initialize CDM source metadata
MERGE INTO cdm_source AS target
USING (
  SELECT 
    'Redox FHIR Pipeline' AS cdm_source_name,
    'REDOX' AS cdm_source_abbreviation,
    'Databricks' AS cdm_holder,
    'FHIR bundles from Redox integration' AS source_description,
    'https://github.com/databricks-industry-solutions' AS source_documentation_reference,
    'FHIR to OMOP streaming pipeline' AS cdm_etl_reference,
    CURRENT_DATE() AS source_release_date,
    CURRENT_DATE() AS cdm_release_date,
    'v5.4' AS cdm_version,
    756265 AS cdm_version_concept_id,
    'v5.0' AS vocabulary_version
) AS source
ON target.cdm_source_name = source.cdm_source_name
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

## LOCATION Table

Geographic locations from FHIR Location and Patient address resources.

In [ ]:
CREATE TABLE IF NOT EXISTS location (
  location_id BIGINT NOT NULL COMMENT 'Unique location identifier'
  ,address_1 STRING COMMENT 'Address line 1'
  ,address_2 STRING COMMENT 'Address line 2'
  ,city STRING COMMENT 'City'
  ,state STRING COMMENT 'State'
  ,zip STRING COMMENT 'ZIP/Postal code'
  ,county STRING COMMENT 'County'
  ,country STRING COMMENT 'Country'
  ,location_source_value STRING COMMENT 'Original location identifier'
  ,latitude DOUBLE COMMENT 'Latitude coordinate'
  ,longitude DOUBLE COMMENT 'Longitude coordinate'
)
USING DELTA
COMMENT 'OMOP CDM Location table - Geographic locations'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'quality' = 'gold'
);

## OBSERVATION_PERIOD Table

Derived from patient encounters - represents periods when patient data was captured.

In [ ]:
DECLARE OR REPLACE VARIABLE create_obs_period_stmt STRING;

SET VARIABLE create_obs_period_stmt = "
CREATE OR REFRESH STREAMING TABLE observation_period (
  observation_period_id BIGINT NOT NULL COMMENT 'Unique observation period identifier'
  ,person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  ,observation_period_start_date DATE NOT NULL COMMENT 'Period start date'
  ,observation_period_end_date DATE NOT NULL COMMENT 'Period end date'
  ,period_type_concept_id BIGINT NOT NULL COMMENT 'Type concept: 32817=EHR'
)
COMMENT 'OMOP CDM Observation Period table - Patient data capture periods'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS
-- Derive observation periods from encounter data
SELECT
  ROW_NUMBER() OVER (ORDER BY person_id, min_date) AS observation_period_id
  ,person_id
  ,min_date AS observation_period_start_date
  ,COALESCE(max_date, min_date) AS observation_period_end_date
  ,32817 AS period_type_concept_id  -- EHR
FROM (
  SELECT
    ABS(HASH(
      COALESCE(
        REGEXP_EXTRACT(subject:reference::STRING, 'Patient/(.+)', 1),
        subject:reference::STRING
      )
    )) AS person_id
    ,MIN(CAST(TRY_CAST(period:start::STRING AS TIMESTAMP) AS DATE)) AS min_date
    ,MAX(COALESCE(
      CAST(TRY_CAST(period:end::STRING AS TIMESTAMP) AS DATE),
      CAST(TRY_CAST(period:start::STRING AS TIMESTAMP) AS DATE)
    )) AS max_date
  FROM STREAM(" || catalog_use || "." || silver_schema || ".encounter)
  WHERE subject:reference IS NOT NULL
    AND period:start IS NOT NULL
  GROUP BY person_id
)
";

SELECT create_obs_period_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_obs_period_stmt;

## DEATH Table

Patient death records from FHIR Patient.deceasedDateTime or deceasedBoolean.

In [ ]:
DECLARE OR REPLACE VARIABLE create_death_stmt STRING;

SET VARIABLE create_death_stmt = "
CREATE OR REFRESH STREAMING TABLE death (
  person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  ,death_date DATE NOT NULL COMMENT 'Date of death'
  ,death_datetime TIMESTAMP COMMENT 'Datetime of death'
  ,death_type_concept_id BIGINT NOT NULL COMMENT 'Type concept: 32817=EHR'
  ,cause_concept_id BIGINT COMMENT 'Cause of death concept'
  ,cause_source_value STRING COMMENT 'Original cause of death code'
  ,cause_source_concept_id BIGINT COMMENT 'Source concept for cause'
)
COMMENT 'OMOP CDM Death table - Patient mortality records'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS
SELECT
  ABS(HASH(COALESCE(id::STRING, patient_uuid))) AS person_id
  ,CAST(TRY_CAST(deceasedDateTime::STRING AS TIMESTAMP) AS DATE) AS death_date
  ,TRY_CAST(deceasedDateTime::STRING AS TIMESTAMP) AS death_datetime
  ,32817 AS death_type_concept_id  -- EHR
  ,NULL AS cause_concept_id
  ,NULL AS cause_source_value
  ,NULL AS cause_source_concept_id
FROM STREAM(" || catalog_use || "." || silver_schema || ".patient)
WHERE deceasedDateTime IS NOT NULL
   OR deceasedBoolean::STRING = 'true'
";

SELECT create_death_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_death_stmt;

In [ ]:
-- Verify CDM source
SELECT * FROM cdm_source;

In [ ]:
-- Verify observation periods
SELECT 
  observation_period_id,
  person_id,
  observation_period_start_date,
  observation_period_end_date,
  DATEDIFF(observation_period_end_date, observation_period_start_date) AS days_observed
FROM observation_period
LIMIT 10;

In [ ]:
-- Death records summary
SELECT 
  COUNT(*) AS death_count,
  MIN(death_date) AS earliest_death,
  MAX(death_date) AS latest_death
FROM death;